In [0]:
import json, urllib.request
from pyspark.sql import functions as F

latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"
LANDING_DIR = "/Volumes/workspace/spb/landing"

def to_bronze(df):
    return(
        df
        .withColumn("version", F.lit(latest))
        .withColumn("ingested_at", F.current_timestamp())
    )

def to_bronze_osm(df):
    return(
        df.withColumn("ingested_at", F.current_timestamp())
    )


(
to_bronze(spark.read.parquet(f"{LANDING_DIR}/places"))
.write.format("delta")
.mode("overwrite")
.saveAsTable("workspace.spb.bronze_places")
)

(
to_bronze(spark.read.parquet(f"{LANDING_DIR}/buildings"))
.write.format("delta")
.mode("overwrite")
.saveAsTable("workspace.spb.bronze_buildings")
)

(
to_bronze(spark.read.parquet(f"{LANDING_DIR}/division_area"))
.write.format("delta")
.mode("overwrite")
.saveAsTable("workspace.spb.bronze_division_area")
)

(
to_bronze_osm(spark.read.parquet(f"{LANDING_DIR}/walk_edges"))
.write.format("delta")
.mode("overwrite")
.saveAsTable("workspace.spb.bronze_walk_edges")
)

(
to_bronze_osm(spark.read.parquet(f"{LANDING_DIR}/walk_nodes"))
.write.format("delta")
.mode("overwrite")
.saveAsTable("workspace.spb.bronze_walk_nodes")
)